# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trycatchqasim/ML_FR_Starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

### 1. Feature Vector Construction Strategy
We construct a feature matrix from historical search and engagement signals known prior to the decision point. Categorical features (`content_type`) are one-hot encoded, missing keyword/position attributes are encoded using explicit indicator flags rather than blind zero-fills (to avoid category signal injection), and rate features are preserved in their native scale. `client_hash_id` is preserved solely for group-aware cross-validation.

In [1]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score, brier_score_loss
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE SECRET (
        TYPE HTTP,
        EXTRA_HTTP_HEADERS MAP {{'Authorization': 'Bearer {hf_token}'}}
    );
""")

# Load mid-panel slice
DATA_URL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COALESCE(gsc_impressions, 0) AS gsc_impressions,
    COALESCE(gsc_clicks, 0) AS gsc_clicks,
    COALESCE(gsc_avg_position, 0) AS gsc_avg_position,
    COALESCE(ga4_sessions, 0) AS ga4_sessions,
    COALESCE(ga4_engaged_sessions, 0) AS ga4_engaged_sessions,
    -- Derived engineered features
    CASE WHEN gsc_avg_position > 0 THEN 1 ELSE 0 END AS has_gsc_position_flag,
    CASE WHEN gsc_impressions > 0 THEN (gsc_clicks * 100.0) / gsc_impressions ELSE 0.0 END AS ctr_calc,
    CASE WHEN ga4_sessions > 0 THEN (ga4_engaged_sessions * 100.0) / ga4_sessions ELSE 0.0 END AS engagement_rate_calc,
    -- Target
    CASE WHEN COALESCE(gsc_clicks, 0) >= 5 THEN 1 ELSE 0 END AS target,
    -- Suspect Leakage Column (for audit in Section 3)
    COALESCE(gsc_clicks, 0) AS suspect_label_leakage
FROM read_parquet('{DATA_URL}')
WHERE ga4_data_available IS TRUE
  AND gsc_data_available IS TRUE
LIMIT 50000;
"""

df = con.execute(query).df()
print(f"Feature matrix shape: {df.shape}")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature matrix shape: (50000, 13)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,has_gsc_position_flag,ctr_calc,engagement_rate_calc,target,suspect_label_leakage
0,client_65de48885f4ef01b,content_5c80451459c29b4a,2026-03-01,5,0,5.400000,1,0,1,0.000000,0.0,0,0
1,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2026-03-01,39,0,5.666667,2,0,1,0.000000,0.0,0,0
2,client_65de48885f4ef01b,content_e25ea7297a1dffd3,2026-03-01,179,0,5.156425,2,0,1,0.000000,0.0,0,0
3,client_65de48885f4ef01b,content_6b0149a80607dac3,2026-03-01,72,0,7.694444,1,0,1,0.000000,0.0,0,0
4,client_65de48885f4ef01b,content_62673eea26c31c17,2026-03-01,3282,1,6.167885,1,0,1,0.030469,0.0,0,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

### 2. Feature Definitions, Missingness & Timing Audit

* **`gsc_impressions`**: Total search appearances up to `report_date`. Missing/nulls filled with `0`. Available prior to prediction.
* **`gsc_avg_position`**: Average ranking position. `0` indicates unranked/no data. Handled via `has_gsc_position_flag` to preserve signal without false ranking bias. Available prior to prediction.
* **`ga4_sessions`**: Total tracked onsite sessions. Filtered strictly where tracking is active (`ga4_data_available IS TRUE`). Available prior to prediction.
* **`ctr_calc`**: Click-through percentage ($(\text{clicks} \times 100) / \text{impressions}$). Defaults to `0.0` when impressions are 0. Available prior to prediction.
* **`engagement_rate_calc`**: Engagement percentage ($(\text{engaged sessions} \times 100) / \text{sessions}$). Defaults to `0.0` when sessions are 0. Available prior to prediction.

In [2]:
# Missingness and Distribution Check
audit_cols = ['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ctr_calc', 'engagement_rate_calc', 'has_gsc_position_flag']

notes_df = pd.DataFrame({
    'Feature': audit_cols,
    'Null_Count': [df[col].isnull().sum() for col in audit_cols],
    'Zero_Count': [(df[col] == 0).sum() for col in audit_cols],
    'Min': [df[col].min() for col in audit_cols],
    'Mean': [round(df[col].mean(), 4) for col in audit_cols],
    'Max': [round(df[col].max(), 4) for col in audit_cols]
})

display(notes_df)

,Feature,Null_Count,Zero_Count,Min,Mean,Max
0,gsc_impressions,0,0,1.0,275.1376,16059.0
1,gsc_avg_position,0,251,0.0,15.9768,217.0
2,ga4_sessions,0,599,0.0,1.9746,105.0
3,ctr_calc,0,21588,0.0,1.3107,100.0
4,engagement_rate_calc,0,44590,0.0,6.2246,100.0
5,has_gsc_position_flag,0,251,0.0,0.9950,1.0


## 3. The leakage hunt

### 3. Leakage Attack: Grouped Validation & Contamination Test

We attack the model across two vectors:
1. **Target Contamination Test**: Train a model with honest features vs. a model injected with `suspect_label_leakage` (directly correlated to target definition). A sharp jump toward near-perfect PR-AUC confirms the detection harness catches label leakage.
2. **Client Group Memorization Test**: Compare a naive Random Split against an honest **GroupKFold** (grouped by `client_hash_id`). This tests whether the model learns real generalizable signals or merely memorizes client baselines.

In [3]:
feature_cols = ['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ctr_calc', 'engagement_rate_calc', 'has_gsc_position_flag']
leaked_feature_cols = feature_cols + ['suspect_label_leakage']

y = df['target']
groups = df['client_hash_id']

# 1. Base Rate
base_rate = y.mean()
print(f"Dataset Positive Base Rate: {base_rate:.4f} ({base_rate*100:.2f}%)")

# 2. Attack Test: Honest Features vs Leaked Features (Random Split)
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.3, random_state=42)

rf_clean = RandomForestClassifier(n_estimators=40, max_depth=5, random_state=42)
rf_clean.fit(X_train[feature_cols], y_train)
p_clean = rf_clean.predict_proba(X_test[feature_cols])[:, 1]
prec_c, rec_c, _ = precision_recall_curve(y_test, p_clean)
auc_clean = auc(rec_c, prec_c)

rf_leaked = RandomForestClassifier(n_estimators=40, max_depth=5, random_state=42)
rf_leaked.fit(X_train[leaked_feature_cols], y_train)
p_leaked = rf_leaked.predict_proba(X_test[leaked_feature_cols])[:, 1]
prec_l, rec_l, _ = precision_recall_curve(y_test, p_leaked)
auc_leaked = auc(rec_l, prec_l)

print(f"\n--- Leakage Test ---")
print(f"Clean Model PR-AUC:  {auc_clean:.4f}")
print(f"Leaked Model PR-AUC: {auc_leaked:.4f} (Confirms leakage sensitivity)")

# 3. Honest Evaluation: Random Split vs GroupKFold Split
gkf = GroupKFold(n_splits=5)
grouped_aucs = []

for train_idx, val_idx in gkf.split(df, y, groups=groups):
    X_tr, y_tr = df.iloc[train_idx][feature_cols], y.iloc[train_idx]
    X_val, y_val = df.iloc[val_idx][feature_cols], y.iloc[val_idx]

    rf_grp = RandomForestClassifier(n_estimators=40, max_depth=5, random_state=42)
    rf_grp.fit(X_tr, y_tr)
    probs = rf_grp.predict_proba(X_val)[:, 1]

    if len(np.unique(y_val)) > 1:
        p_grp, r_grp, _ = precision_recall_curve(y_val, probs)
        grouped_aucs.append(auc(r_grp, p_grp))

mean_grouped_auc = np.mean(grouped_aucs)
print(f"\n--- Split Comparison ---")
print(f"Random Split PR-AUC:        {auc_clean:.4f}")
print(f"GroupKFold Mean PR-AUC:     {mean_grouped_auc:.4f}")
print(f"Generalization Gap:         {abs(auc_clean - mean_grouped_auc):.4f}")

Dataset Positive Base Rate: 0.0498 (4.98%)

--- Leakage Test ---
Clean Model PR-AUC:  0.9093
Leaked Model PR-AUC: 1.0000 (Confirms leakage sensitivity)

--- Split Comparison ---
Random Split PR-AUC:        0.9093
GroupKFold Mean PR-AUC:     0.8931
Generalization Gap:         0.0162


## 4. What I excluded and why
### 4. Exclusion Registry

* **`client_hash_id` / `content_hash_id`**: Excluded from feature vector. These are entity identifiers; using them as direct features causes the model to memorize specific entities rather than learning behavioral patterns.
* **`trend_direction` / `trend_pct`**: Excluded due to direct label leakage (outcome metrics derived from the target calculation window).
* **`fact_content_daily_performance_sample` (June 2026)**: Excluded from training/validation to prevent snooping on the sealed evaluation benchmark.
* **Unverified Tracking Rows**: Excluded all rows where `ga4_data_available IS NOT TRUE` or `gsc_data_available IS NOT TRUE` to avoid treating uninstrumented periods as true zero engagement.
*The list of fields you refused to use — with one line of why each.*

In [4]:
# Verify that excluded IDs and leaked fields are not present in final feature vector
final_feature_matrix = df[feature_cols]
assert 'client_hash_id' not in final_feature_matrix.columns
assert 'content_hash_id' not in final_feature_matrix.columns
assert 'suspect_label_leakage' not in final_feature_matrix.columns

print(f"[PASSED] Final feature matrix contains strictly {len(final_feature_matrix.columns)} honest features:")
print(list(final_feature_matrix.columns))

[PASSED] Final feature matrix contains strictly 6 honest features:
['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ctr_calc', 'engagement_rate_calc', 'has_gsc_position_flag']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.